# Anthive Client Example

This notebook demonstrates how to use the Anthive Client to query single-cell expression databases.

## Setup: Install Required Packages

Install required packages with micropip (this may take a minute on first run).

In [ ]:
import micropip
await micropip.install(['requests', 'pandas', 'matplotlib', 'seaborn'])
print("✓ Packages installed")

## 1. Import Libraries

The required packages are now available.

In [ ]:
# Import antclient (available in this JupyterLite environment)
from antclient import AnthiveClient

# Import analysis libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Libraries imported successfully")

## 2. Connect to Anthive API

The client auto-detects the API URL in browser environments.

In [ ]:
# Create client - get current origin from browser
try:
    import js
    api_url = f"{js.location.origin}/api"
except:
    api_url = None  # Will auto-detect

client = AnthiveClient(api_url)
print(f"Connected to: {client.base_url}")

## 3. Discover Databases

In [ ]:
# List all databases
databases = client.get_databases(format='dataframe')
print(f"Found {len(databases)} databases")

# Display database info
databases[['id', 'n_cells', 'n_genes', 'title']].head()

## 4. Explore a Database

In [ ]:
# Select first database
db_id = databases.iloc[0]['id']
print(f"Using database: {db_id}")

# Get database info
info = client.get_database_info(db_id)
print(f"Cells: {info['n_cells']:,}")
print(f"Genes: {info['n_genes']:,}")
print(f"Layers: {info['layers']}")
print(f"Embeddings: {info['embeddings']}")

# Determine which layer to use (prefer "X", fallback to first available)
if 'X' in info['layers']:
    layer = 'X'
elif info['layers']:
    layer = info['layers'][0]
else:
    layer = 'X'  # fallback default

print(f"\nUsing layer: {layer}")

## 5. Search Genes

In [ ]:
# Search for genes
genes = client.search_genes(db_id, q="CD", limit=10)
print(f"Genes matching 'CD': {genes}")

## 6. Get Cell Data

In [ ]:
# Get expression data for specific genes
df = client.get_cells(
    db_id,
    genes=["CD4", "CD8A"],
    metadata=["cell_type"],
    layer=layer,  # Use detected layer
    limit=1000,
    format='dataframe'
)

print(f"Retrieved {len(df)} cells")
df.head()

## 7. Visualize Expression

In [ ]:
# Plot gene expression
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
df['CD4'].hist(bins=50, ax=ax1)
ax1.set_xlabel('CD4 Expression')
ax1.set_ylabel('Count')
ax1.set_title('CD4 Expression Distribution')

# Scatter plot
ax2.scatter(df['CD4'], df['CD8A'], alpha=0.5, s=10)
ax2.set_xlabel('CD4')
ax2.set_ylabel('CD8A')
ax2.set_title('CD4 vs CD8A Expression')

plt.tight_layout()
plt.show()

## 8. Get Embeddings

In [ ]:
# Get UMAP embedding if available
embeddings = client.get_embeddings(db_id)
print(f"Available embeddings: {embeddings}")

if embeddings:
    umap_df = client.get_embedding_data(
        db_id,
        embeddings[0],
        n_dims=2,
        limit=5000,
        format='dataframe'
    )
    
    # Plot UMAP
    plt.figure(figsize=(8, 6))
    plt.scatter(umap_df['dim_0'], umap_df['dim_1'], s=1, alpha=0.5)
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.title(f'{embeddings[0]} Embedding')
    plt.show()

## 9. SQL Queries

For advanced queries, you can execute SQL directly.

In [ ]:
# Execute SQL query
sql_df = client.execute_sql(
    db_id,
    "SELECT * FROM obscat LIMIT 100",
    format='dataframe'
)

print(f"Retrieved {len(sql_df)} rows")
sql_df.head()

## Next Steps

- Explore more genes and cell types
- Combine expression data with metadata filters
- Perform statistical analysis with statsmodels
- Create custom visualizations with seaborn

See the [Anthive Client documentation](../doc/ANTCLIENT_README.md) for more examples.